# Case Técnico — Engenharia de Dados

##LOAD - Preparação do ambiente

Nesta etapa os arquivos CSV são carregados em DataFrames para permitir a execução das consultas SQL por meio do pandasql.

Os dados não são versionados neste repositório e devem estar disponíveis em `../data/`.

In [115]:
#pip install pandasql

In [116]:
import pandas as pd
from pandasql import sqldf

In [117]:
pysqldf = lambda q: sqldf(q, globals())

In [118]:
buyers = pd.read_csv('../data/buyers.csv', sep=',')
order_items = pd.read_csv('../data/order_items.csv', sep=',')
orders = pd.read_csv('../data/orders.csv', sep=',')
payments = pd.read_csv('../data/payments.csv', sep=',')
products = pd.read_csv('../data/products.csv', sep=',')
sellers = pd.read_csv('../data/sellers.csv', sep=',')

In [119]:
print("ORDERS")
display(orders.head())

print("ORDER_ITEMS")
display(order_items.head())

print("PRODUCTS")
display(products.head())

print("SELLERS")
display(sellers.head())

print("BUYERS")
display(buyers.head())

print("PAYMENTS")
display(payments.head())

ORDERS


,id,seller_id,buyer_id,status,created_at,total_value
0,1,114,2806,delivered,2023-08-22 05:25:59,1987.16
1,2,90,2586,processing,2024-07-19 03:57:54,24074.93
2,3,96,1849,completed,2024-06-14 22:16:15,2416.42
3,4,10,116,processing,2024-07-11 04:46:30,3871.48
4,5,16,2195,delivered,2023-09-07 20:06:40,15367.65


ORDER_ITEMS


,id,order_id,product_id,qty,unit_price,discount
0,1,1,500,6,240.06,117.48
1,2,1,778,6,126.47,94.54
2,3,2,346,39,492.08,98.86
3,4,2,248,20,278.55,588.33
4,5,3,701,9,298.75,272.33


PRODUCTS


...

,id,name,state,plan,created_at
0,1,Santos & Silva Atacado Distribuidora,BA,free,2023-04-17 00:16:11
1,2,Souza & Souza Comércio Distribuidora,DF,premium,2022-09-09 22:06:21
2,3,Santos & Almeida Distribuidora Distribuidora,AM,basic,2022-12-16 08:31:25
3,4,Nascimento & Ferreira Alimentos Distribuidora,GO,basic,2023-05-30 23:12:37
4,5,Silva & Santos Suprimentos Distribuidora,BA,premium,2023-03-06 10:29:33


BUYERS


,id,name,city,state,segment,created_at
0,1,Nascimento & Costa Distribuidora Mercearia,Recife,DF,supermarket,2023-04-03 03:46:36
1,2,Lima & Almeida Atacado Mercearia,Rio de Janeiro,PE,convenience,2023-08-05 01:36:12
2,3,Ferreira & Costa Grupo Mercearia,São Paulo,PR,grocery,2023-07-10 11:37:01
3,4,Lima & Almeida Comércio Supermercado,Campo Grande,MT,convenience,2023-10-13 01:08:52
4,5,Ferreira & Ferreira Comércio Mercearia,São Paulo,PB,convenience,2024-04-20 00:23:27


PAYMENTS


,id,order_id,paid_at,amount,method,status
0,1,1,2023-08-23 19:25:59,1987.16,transfer,paid
1,2,2,2024-07-20 17:57:54,24074.93,boleto,paid
2,3,3,2024-06-16 10:16:15,2416.42,transfer,paid
3,4,4,2024-07-11 11:46:30,3871.48,boleto,paid
4,5,5,2023-09-08 00:06:40,15367.65,transfer,paid


##TRANSFORM - Preparação dos dados

A única conversão explícita aplicada antes das consultas é `orders.created_at` para datetime.

Demais verificações de schema, NULL, unicidade e integridade referencial não são automatizadas nesta versão do projeto.

In [120]:
# Converte a coluna para datetime real
orders['created_at'] = pd.to_datetime(orders['created_at'])
orders.dtypes

id                      int64
seller_id               int64
buyer_id                int64
status                    str
created_at     datetime64[us]
total_value           float64
dtype: object

DESAFIO 1)
Cada questão SQL resolve um problema real que está travando as decisões da empresa. 
O time financeiro reclama que o total de faturamento mensal varia dependendo de quem puxa o 
relatório. Você descobre que pedidos com status 'cancelled' e 'refunded' estão sendo incluídos em 
alguns relatórios. 
Escreva uma query que retorne o faturamento bruto mensal dos últimos 12 meses, 
considerando apenas pedidos com status 'completed' ou 'delivered'. Inclua também a 
quantidade de pedidos e o ticket médio de cada mês. Ordene do mês mais recente ao mais 
antigo.

In [121]:
# Desafio 1

dt_end = orders['created_at'].max()
dt_end

Timestamp('2024-11-29 23:37:25')

Os dados de pedidos vão até novembro/2024. Irei calcular a data de inicio para ter os 12 meses (mês atual +11) fora do SQL.

In [122]:
dt_start = (orders['created_at'].max().replace(day=1)- pd.DateOffset(months=11))
dt_start

Timestamp('2023-12-01 23:37:25')

Primeiro, tratei possíveis duplicidades na tabela de pedidos, mantendo apenas o registro mais recente para cada chave de negócio.

## 📊 Definição de Faturamento

Para este case, a coluna `orders.total_value` representa o valor final pago pelo cliente após a aplicação dos descontos.

A regra de negócio segue a fórmula:

$$\text{Valor Bruto do Pedido} - \text{Descontos} = \text{total\_value}$$

Dessa forma, `total_value` é utilizado como o **faturamento da venda**, representando o valor efetivamente pago pelo cliente ao vendedor.

> 📝 **Nota para o Desafio 1:** O faturamento mensal é obtido por meio da soma de `total_value` (vl_net_sale) de todos os pedidos considerados válidos.



In [123]:
# Desafio 1
query = f"""
WITH order_his AS (
--NO DUPLICATES:
    SELECT created_at dt_order, id cd_order, seller_id cd_customer_branch, buyer_id cd_local_store, status ds_status, total_value vl_net_sale FROM (
        SELECT *, ROW_NUMBER() OVER (PARTITION BY id ORDER BY created_at DESC) row_number FROM (
            SELECT *
            FROM orders
            WHERE 1 = 1 
                AND created_at <= CURRENT_DATE
        ) WHERE 1 = 1 
    ) WHERE row_number = 1
)
--select * from order_his

SELECT
    strftime('%Y-%m', dt_order) AS year_month
,   SUM(vl_net_sale) AS vl_net_sale
,   COUNT(cd_order) AS qt_order
,   AVG(vl_net_sale) AS vl_avg_order
FROM order_his
WHERE 1=1
  AND ds_status IN ('completed', 'delivered') --apenas pedidos concluídos ou entregues
  AND dt_order >= '{dt_start.strftime('%Y-%m-%d')}' --apenas pedidos a partir do dia 1 de 12 meses atrás
GROUP BY strftime('%Y-%m', dt_order) -- agrupando por ano e mês
ORDER BY year_month DESC
"""

resultado_desafio1 = pysqldf(query)
resultado_desafio1

,year_month,vl_net_sale,qt_order,vl_avg_order
0,2024-11,56710238.63,3643,15566.906020
1,2024-10,62493439.13,3863,16177.437000
2,2024-09,60274462.50,3779,15949.844536
3,2024-08,60102539.82,3743,16057.317612
4,2024-07,59769406.95,3862,15476.283519
5,2024-06,58797774.37,3644,16135.503395
6,2024-05,60730315.72,3848,15782.306580
7,2024-04,57779312.59,3653,15816.948423
8,2024-03,61525913.64,3877,15869.464442
9,2024-02,57926376.26,3628,15966.476367


## 🗓️ Janela Temporal

No Desafio 1, a análise considera uma janela de **exatamente 12 meses**, tomando como referência o período mais recente disponível na base de dados.

Para garantir a integridade da análise de faturamento, são considerados apenas os pedidos com os seguintes status:
* `completed`
* `delivered`

A agregação final é realizada estritamente no nível mensal, onde:
* **1 linha = 1 mês**


DESAFIO 2)
A diretora comercial quer um ranking dos 10 sellers com maior crescimento de GMV entre o 
trimestre atual e o anterior, mas só quer ver sellers que tiveram pelo menos 50 pedidos em ambos 
os trimestres (para evitar distorção de sellers novos ou inativos). 
Escreva a query que resolve esse problema, exibindo: nome do seller, estado, GMV do 
trimestre anterior, GMV do trimestre atual e o percentual de crescimento. Ordene pelo maior 
crescimento.

In [124]:
# Desafio 2

dt_end = orders['created_at'].max()
dt_end

Timestamp('2024-11-29 23:37:25')

Considerando então que o trimestre atual é o da maior data (2024-11-29), ou seja, out/nov/dez de 2024. Será comparado esse trimestre com o anterior (jul/ago/set de 2024).

In [125]:
nr_year = dt_end.year ##ano mais recente
nr_quarter = (dt_end.month - 1) // 3 + 1 ##transforma o mês em número do trimestre:

if nr_quarter == 1: ##se for o primeiro trimestre, o trimestre anterior é o quarto do ano anterior
    nr_year_previous = nr_year - 1
    nr_quarter_previous = 4
else: 
    nr_year_previous = nr_year
    nr_quarter_previous = nr_quarter - 1

print(nr_year, nr_quarter)
print(nr_year_previous, nr_quarter_previous)

2024 4
2024 3


O objetivo do trecho acima é descobrir automaticamente o ano e o trimestre do último dado disponível (dt_end) e também identificar o trimestre imediatamente anterior, sem precisar informar manualmente.

Primeiro, tratei possíveis duplicidades nas tabelas de sellers e pedidos, mantendo apenas o registro mais recente para cada chave de negócio.

Depois calculado o trimestre e get no ano daquele registro, para em seguida calcular valor e quantidade por trimestre de cada vendedor em cada trimestre/ano. Por fim, LAG e percentual de crescimento.

In [126]:
# Desafio 2
query = f"""
WITH customer_branch AS (
--NO DUPLICATES:
    SELECT id cd_customer_branch, name nm_customer_branch, state cd_uf, plan ds_plan, created_at dt_processing FROM (
        SELECT *, ROW_NUMBER() OVER (PARTITION BY id ORDER BY created_at DESC) row_number FROM (
            SELECT *
            FROM sellers
            WHERE 1=1
                AND created_at <= CURRENT_DATE
        ) WHERE 1 = 1 
    ) WHERE row_number = 1
)
--select * from customer_branch

, order_his AS (
--NO DUPLICATES:
    SELECT created_at dt_order, id cd_order, seller_id cd_customer_branch, buyer_id cd_local_store, status ds_status, total_value vl_net_sale FROM (
        SELECT *, ROW_NUMBER() OVER (PARTITION BY id ORDER BY created_at DESC) row_number FROM (
            SELECT *
            FROM orders
            WHERE 1 = 1 
                AND created_at <= CURRENT_DATE
        ) WHERE 1 = 1 
    ) WHERE row_number = 1
)
--select * from order_his 

, order_transform AS (
    SELECT
        cd_customer_branch
    ,   cd_order
    ,   vl_net_sale
    ,   CAST((CAST(strftime('%m', dt_order) AS INTEGER) + 2) / 3 AS INTEGER) AS nr_quarter --calcula numero do trimestre a partir do mês
    ,   CAST(strftime('%Y', dt_order) AS INTEGER) AS nr_year --get do ano a partir da data
    FROM order_his
    WHERE 1=1
        AND ds_status IN ('completed', 'delivered') --do not consider cancelled/processing/refunded orders
)
--select * from order_transform

, order_quarter AS (
    SELECT
        cd_customer_branch 
    ,   nr_year
    ,   nr_quarter
    ,   SUM(vl_net_sale) AS vl_gmv_quarter --Valor total de vendas do trimestre
    ,   COUNT(cd_order) AS qt_order_quarter --Quantidade de pedidos do trimestre
    FROM order_transform
    WHERE 1=1
        AND ((nr_year = {nr_year} AND nr_quarter = {nr_quarter}) OR (nr_year = {nr_year_previous} AND nr_quarter = {nr_quarter_previous}))   --apenas do periodo que interessa     
    GROUP BY cd_customer_branch, nr_year, nr_quarter --agrupando por vendedor, ano e trimestre
)
--select * from order_quarter

, order_lag AS (
    SELECT
        cd_customer_branch 
    ,   nr_year
    ,   nr_quarter
    ,   vl_gmv_quarter
    ,   qt_order_quarter
    ,   LAG(vl_gmv_quarter) OVER (PARTITION BY cd_customer_branch ORDER BY nr_year, nr_quarter) AS vl_gmv_previous_quarter --Valor total de vendas do trimestre anterior
    ,   LAG(qt_order_quarter) OVER (PARTITION BY cd_customer_branch ORDER BY nr_year, nr_quarter) AS qt_order_previous_quarter --Quantidade de pedidos do trimestre anterior
    FROM order_quarter
    WHERE 1=1
)
--select * from order_lag


SELECT 
--    order_lag.cd_customer_branch,
   customer_branch.nm_customer_branch
,   customer_branch.cd_uf
--,   order_lag.nr_year AS nr_year
--,   order_lag.nr_quarter AS nr_quarter  
--,   order_lag.qt_order_previous_quarter
,   order_lag.vl_gmv_previous_quarter 
--,   order_lag.qt_order_quarter 
,   order_lag.vl_gmv_quarter
,   CASE
        WHEN order_lag.vl_gmv_previous_quarter = 0 THEN NULL 
        ELSE ROUND(( order_lag.vl_gmv_quarter -  order_lag.vl_gmv_previous_quarter) /  order_lag.vl_gmv_previous_quarter * 100, 2)
    END pc_growth_gmv  --percentual de crescimento do GMV do trimestre atual em relação ao trimestre anterior (tratamento divisão por zero aplicado)

FROM order_lag 
INNER JOIN customer_branch ON  order_lag.cd_customer_branch = customer_branch.cd_customer_branch
WHERE 1=1
    AND order_lag.nr_year = '{nr_year}' --apenas do ano do trimestre mais recente
    AND order_lag.nr_quarter = '{nr_quarter}' --apenas do trimestre mais recente
    AND order_lag.qt_order_quarter >= 50  -- mínimo de 50 pedidos nos dois trimestres
    AND order_lag.qt_order_previous_quarter >= 50  -- mínimo de 50 pedidos nos dois trimestres
ORDER BY pc_growth_gmv DESC NULLS LAST --ordenando do maior percentual de crescimento para o menor, colocando os nulos por último
LIMIT 10 -- apenas os 10 primeiros resultados

"""

resultado_desafio2 = pysqldf(query)
resultado_desafio2

,nm_customer_branch,cd_uf,vl_gmv_previous_quarter,vl_gmv_quarter,pc_growth_gmv
0,Rodrigues & Almeida Atacado Distribuidora,MG,876302.72,856460.38,-2.26
1,Costa & Silva Alimentos Distribuidora,BA,1017817.89,976148.08,-4.09
2,Costa & Santos Suprimentos Distribuidora,DF,876787.95,807935.32,-7.85
3,Nascimento & Souza Alimentos Distribuidora,SP,1031166.40,933683.69,-9.45
4,Almeida & Almeida Alimentos Distribuidora,RS,1023319.44,923462.61,-9.76
5,Nascimento & Almeida Alimentos Distribuidora,SP,949398.56,847228.59,-10.76
6,Lima & Almeida Grupo Distribuidora,CE,1052368.65,938858.41,-10.79
7,Almeida & Souza Mercado Distribuidora,PR,1095325.17,973049.07,-11.16
8,Costa & Santos Atacado Distribuidora,MG,1106858.77,954140.51,-13.80
9,Almeida & Rodrigues Distribuidora Distribuidora,RJ,1102447.24,911827.61,-17.29


Foi validado que no período do dataset nenhum seller preencheu o requisito de crescimento positivo, listando assim os que tiveram menor queda. Isso se deve ao trimestre atual ainda não ter fechado (ultima data de referencia dia 29/11/2024).

DESAFIO 3) 
O time de fraudes suspeita que alguns sellers estão aplicando descontos abusivos para inflar volume artificialmente. Você precisa encontrar todos os pedidos onde o desconto total (soma dos descontos dos itens) representa mais de 40% do valor bruto do pedido, listando também o seller responsável e a data do pedido. Exclua pedidos cancelados. 
Escreva a query e explique brevemente o raciocínio por trás dela. 

Primeiro, tratei possíveis duplicidades nas tabelas de sellers, pedidos e itens, mantendo apenas o registro mais recente para cada chave de negócio.

-CTEs utilizadas:

--customer_branch: deduplica a tabela de sellers para evitar duplicidades no relacionamento e disponibiliza as informações dos distribuidores, como nome, estado e plano.
--order_item: deduplica os itens dos pedidos por order_id + product_id, mantendo o registro mais recente.
--order_his: deduplica os pedidos por id, mantendo o registro mais recente.

Depois, na CTE order_not_cancelled, relacionei os pedidos aos respectivos itens e aos sellers. Nessa etapa, excluí os pedidos com status cancelled e refunded, pois eles não devem participar da análise.

Em seguida, calculei o valor bruto dos itens do pedido como:

    SUM(qt_order_item * vl_price_unit)

e o desconto total como:

    SUM(vl_discount_item)

Por fim, comparei o desconto total com 40% do valor bruto do pedido. Foram considerados suspeitos os pedidos em que:

    vl_discount_total > 0.4 * vl_gross_sale

Como resultado, foram identificados 853 pedidos que atendem ao critério de suspeita de desconto abusivo.

In [127]:
#Desafio 3
query = """
WITH customer_branch AS (
--NO DUPLICATES:
    SELECT id cd_customer_branch, name nm_customer_branch, state cd_uf, plan ds_plan, created_at dt_processing FROM (
        SELECT *, ROW_NUMBER() OVER (PARTITION BY id ORDER BY created_at DESC) row_number FROM (
            SELECT *
            FROM sellers
            WHERE 1=1
                AND created_at <= CURRENT_DATE
        ) WHERE 1 = 1 
    ) WHERE row_number = 1
)
--select * from customer_branch

, order_item AS (
--NO DUPLICATES:
    SELECT order_id cd_order, product_id cd_product, qty qt_order_item, unit_price vl_price_unit, discount vl_discount_item FROM (
        SELECT *, ROW_NUMBER() OVER (PARTITION BY order_id, product_id ORDER BY id DESC) row_number FROM (
            SELECT *
            FROM order_items
            WHERE 1 = 1 
        ) WHERE 1 = 1 
    ) WHERE row_number = 1
)
--select * from order_item

, order_his AS (
--NO DUPLICATES:
    SELECT created_at dt_order, id cd_order, seller_id cd_customer_branch, buyer_id cd_local_store, status ds_status, total_value vl_net_sale FROM (
        SELECT *, ROW_NUMBER() OVER (PARTITION BY id ORDER BY created_at DESC) row_number FROM (
            SELECT *
            FROM orders
            WHERE 1 = 1 
                AND created_at <= CURRENT_DATE
        ) WHERE 1 = 1 
    ) WHERE row_number = 1
)
--select * from order_his

, order_not_cancelled AS (
SELECT order_his.dt_order
    , order_item.cd_order
    , SUM(qt_order_item * vl_price_unit) AS vl_gross_sale
    , SUM(vl_discount_item) AS vl_discount_total
    , order_his.vl_net_sale AS vl_net_sale
    , order_his.cd_customer_branch
    , customer_branch.nm_customer_branch 
FROM order_item
INNER JOIN order_his ON order_item.cd_order = order_his.cd_order
INNER JOIN customer_branch ON order_his.cd_customer_branch = customer_branch.cd_customer_branch
WHERE 1=1
    AND order_his.ds_status NOT IN ('cancelled', 'refunded') -- not consider cancelled/refunded orders
GROUP BY order_item.cd_order
,   order_his.dt_order
,   order_his.vl_net_sale
,   order_his.cd_customer_branch
,   customer_branch.nm_customer_branch
)

--select * from order_not_cancelled


SELECT * FROM order_not_cancelled WHERE 1=1 AND vl_discount_total > (0.4 * vl_gross_sale)


"""

resultado_desafio3 = pysqldf(query)
resultado_desafio3

,dt_order,cd_order,vl_gross_sale,vl_discount_total,vl_net_sale,cd_customer_branch,nm_customer_branch
0,2024-10-03 00:16:12.000000,30,2284.33,1048.45,1235.88,89,Oliveira & Santos Distribuidora Distribuidora
1,2023-10-11 05:43:04.000000,123,1849.23,751.58,1097.65,55,Souza & Almeida Suprimentos Distribuidora
2,2023-11-25 04:36:24.000000,166,29120.15,15424.13,13696.02,89,Oliveira & Santos Distribuidora Distribuidora
3,2024-03-27 10:22:13.000000,210,26623.92,12146.28,14477.64,99,Santos & Almeida Suprimentos Distribuidora
4,2024-11-02 03:11:05.000000,274,13611.57,6781.87,6829.70,13,Costa & Costa Atacado Distribuidora
...,...,...,...,...,...,...,...
848,2024-07-03 07:13:43.000000,79280,11946.48,6477.38,5469.10,45,Oliveira & Costa Comércio Distribuidora
849,2024-09-23 04:33:12.000000,79652,43166.03,21299.77,21866.26,113,Costa & Oliveira Atacado Distribuidora
850,2023-09-16 21:47:42.000000,79727,15998.66,7394.76,8603.90,13,Costa & Costa Atacado Distribuidora
851,2024-05-24 19:33:26.000000,79761,23536.28,12827.82,10708.46,99,Santos & Almeida Suprimentos Distribuidora


Desafio 4) 
Existe um produto com comportamento estranho: ele tem um volume de vendas alto, mas nunca 
aparece como item mais vendido dentro de nenhum pedido (nunca é o item de maior valor num 
pedido). Encontre todos os produtos que se encaixam nessa descrição: total de unidades vendidas maior que 1.000, mas que em nenhum pedido foram o item de maior valor unitário. 
Escreva a query.  
Dica: window functions podem ser suas aliadas aqui e avalie os resultados e viabilidade da 
análise e faça os questionamentos necessários. 

Primeiro, tratei possíveis duplicidades em order_items, mantendo o registro mais recente para cada combinação de order_id + product_id.
Depois tratei possíveis duplicidades na tabela de pedidos (order_his), mantendo apenas o registro mais recente para cada chave de negócio.

Na order_not_cancelled, fiz a junção da order_item com a order_his, filtrando apenas os pedidos validos (ou seja, excluindo os cancelados ou reembolsados).

A partir da order_not_cancelled, usei as CTEs a seguir para encontrar os produtos conforme as regras de negocio do enunciado.

Na CTE rank_item_price, utilizei RANK() para identificar, dentro de cada pedido, os produtos com maior valor unitário. O uso de RANK() permite considerar empates, caso dois ou mais produtos tenham o mesmo maior valor unitário.

Na CTE sold_item, calculei o total de unidades vendidas por produto.

Por fim, utilizei um LEFT JOIN entre os produtos vendidos e os produtos que já foram classificados como maior valor unitário em algum pedido. Mantive apenas os produtos que nunca apareceram como maior valor unitário e que possuem mais de 1.000 unidades vendidas.

Assim, o resultado identifica os produtos com alto volume de vendas, mas que nunca foram o item de maior valor unitário dentro de um pedido. 

In [128]:
# Desafio 4
query = """
WITH order_item AS (
--NO DUPLICATES:
    SELECT order_id cd_order, product_id cd_product, qty qt_order_item, unit_price vl_price_unit, discount vl_discount_item FROM (
        SELECT *, ROW_NUMBER() OVER (PARTITION BY order_id, product_id ORDER BY id DESC) row_number FROM (
            SELECT *
            FROM order_items
            WHERE 1 = 1 
        ) WHERE 1 = 1 
    ) WHERE row_number = 1
)
--select * from order_item

, order_his AS (
--NO DUPLICATES:
    SELECT created_at dt_order, id cd_order, seller_id cd_customer_branch, buyer_id cd_local_store, status ds_status, total_value vl_net_sale FROM (
        SELECT *, ROW_NUMBER() OVER (PARTITION BY id ORDER BY created_at DESC) row_number FROM (
            SELECT *
            FROM orders
            WHERE 1 = 1 
                AND created_at <= CURRENT_DATE
        ) WHERE 1 = 1 
    ) WHERE row_number = 1
)
--select * from order_his

, order_not_cancelled AS (
SELECT order_item.*
FROM order_item
INNER JOIN order_his ON order_item.cd_order = order_his.cd_order
WHERE 1=1 
    AND order_his.ds_status NOT IN ('cancelled', 'refunded') -- not consider cancelled/refunded orders
)

--select * from order_not_cancelled

, rank_item_price AS (
    SELECT * FROM (
        SELECT
            cd_order
        ,   cd_product
        ,   qt_order_item
        ,   vl_price_unit
        ,   RANK() OVER (PARTITION BY cd_order ORDER BY vl_price_unit DESC) AS rank_price --rank the items in each order by unit price, highest first
        FROM order_not_cancelled
        WHERE 1=1) WHERE rank_price = 1 --maior valor unitário do pedido
)
--select * from rank_item_price


, sold_item AS (
    SELECT cd_product
    ,   SUM(qt_order_item) AS qt_sold_un
    FROM order_not_cancelled
    GROUP BY cd_product
)
--select * from sold_item 


SELECT sold_item.cd_product 
    ,   sold_item.qt_sold_un
    ,   rank_item_price.cd_order
FROM sold_item 
LEFT JOIN rank_item_price ON sold_item.cd_product = rank_item_price.cd_product
WHERE 1=1 
    AND rank_item_price.cd_product IS NULL -- mas que em nenhum pedido foram o item de maior valor unitário
    AND sold_item.qt_sold_un > 1000 --    total de unidades vendidas > 1.000

"""

resultado_desafio4 = pysqldf(query)
resultado_desafio4

,cd_product,qt_sold_un,cd_order


Resultado: não foram identificados produtos que atendam simultaneamente aos dois critérios analisados: mais de 1.000 unidades vendidas e nunca terem sido o item de maior valor unitário em nenhum pedido. Portanto, o resultado da consulta foi vazio.